In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

X_train = pd.read_csv('../data/processed/X_train.csv', index_col=0)
X_val = pd.read_csv('../data/processed/X_val.csv', index_col=0)

y_train = pd.read_csv('../data/processed/y_train.csv', index_col=0)
y_val = pd.read_csv('../data/processed/y_val.csv', index_col=0)


sparse_cols = ['pool_qc', 'misc_feature', 'alley', 'garage_yr_blt']
X_train = X_train.drop(columns=sparse_cols, errors='ignore')
X_val = X_val.drop(columns=sparse_cols, errors='ignore')

numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_cols),
        ('cat', cat_pipeline, categorical_cols)
    ]
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.6,
        random_state=42
    ))
])

model_pipeline.fit(X_train, y_train)

y_pred_log = model_pipeline.predict(X_val)

y_val_dollars = np.expm1(y_val)
y_pred_dollars = np.expm1(y_pred_log)

mae = mean_absolute_error(y_val_dollars, y_pred_dollars)
rmse = np.sqrt(mean_squared_error(y_val_dollars, y_pred_dollars))
r2 = r2_score(y_val_dollars, y_pred_dollars)

print(f"Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"R² Variance Score: {r2:.4f}")

Mean Absolute Error (MAE): $12,362.91
Root Mean Squared Error (RMSE): $19,530.61
R² Variance Score: 0.9433
